In [1]:
#Imports

import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
#Configuração do Mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")

mlflow.set_experiment("Linear_Regression")

mlflow.sklearn.autolog()

In [3]:
#Abrir o dataset

df = pd.read_csv("../DATASET/dataset_expandido.csv")

In [4]:
#Informação relativa ao dataset

print(df.shape)

df.info

(165437, 117)


<bound method DataFrame.info of         ResponseId                                         MainBranch    Age  \
0                1                     I am a developer by profession    <18   
1                2                     I am a developer by profession  35-44   
2                3                     I am a developer by profession  45-54   
3                4                              I am learning to code  18-24   
4                5                     I am a developer by profession  18-24   
...            ...                                                ...    ...   
165432       42848                     I am a developer by profession  25-34   
165433       59527                     I am a developer by profession  45-54   
165434       57523  I am not primarily a developer, but I write co...  35-44   
165435       31965                     I am a developer by profession  25-34   
165436       22750  I used to be a developer by profession, but no...  25-34   

       

In [5]:
#Converter o YearsCode para numérico

df["YearsCodePro_Num"] = df["YearsCodePro"]

df["YearsCodePro_Num"] = df["YearsCodePro_Num"].replace({
    "Less than 1 year": 0,
    "More than 50 years": 51,
    "Sem dado": 0
})

df["YearsCodePro_Num"] = pd.to_numeric(
    df["YearsCodePro_Num"],
    errors="coerce"
)

In [6]:
#Definir quais as nossas features e o nosso target

# ============================================================
# Feature Selection
# ============================================================

features = [
    "YearsCodePro_Num",
    "WorkExp",
    "Age_Code"
]

target = "ConvertedCompYearly"

In [7]:
#Criação de um dataframe apenas com as features e o target

df_linear = df[features + [target]].copy()

print(df_linear.head())

print(df_linear.info())

   YearsCodePro_Num  WorkExp  Age_Code  ConvertedCompYearly
0                 0      9.0         1              65000.0
1                17     17.0         4              65000.0
2                27      9.0         5              65000.0
3                 0      9.0         2              65000.0
4                 0      9.0         2              65000.0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 165437 entries, 0 to 165436
Data columns (total 4 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   YearsCodePro_Num     165437 non-null  int64  
 1   WorkExp              165437 non-null  float64
 2   Age_Code             165437 non-null  int64  
 3   ConvertedCompYearly  165437 non-null  float64
dtypes: float64(2), int64(2)
memory usage: 5.0 MB
None


In [8]:
#Função para as Experiências para a Regressão Linear

def run_linear_experiment(
    run_name,
    X,
    y,
    random_state=42,
    test_size=0.2,
    use_half_data=False
):

    # Reduce dataset size
    if use_half_data:
        X, _, y, _ = train_test_split(
            X,
            y,
            test_size=0.5,
            random_state=random_state
        )

    # Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state
    )

    # Model
    with mlflow.start_run(run_name=run_name):

        lr = LinearRegression()

        lr.fit(X_train, y_train)

        # Predictions
        y_pred = lr.predict(X_test)

        # Metrics
        mae = mean_absolute_error(y_test, y_pred)

        mse = mean_squared_error(y_test, y_pred)

        rmse = np.sqrt(mse)

        r2 = r2_score(y_test, y_pred)

        score = lr.score(X_test, y_test)

        # Parameters
        mlflow.log_param("random_state", random_state)

        mlflow.log_param("test_size", test_size)

        mlflow.log_param("use_half_data", use_half_data)

        mlflow.log_param("n_features", X.shape[1])

        # Metrics
        mlflow.log_metric("mae", mae)

        mlflow.log_metric("mse", mse)

        mlflow.log_metric("rmse", rmse)

        mlflow.log_metric("r2", r2)

        mlflow.log_metric("score", score)

        # Results
        print(run_name)

        print("MAE:", mae)

        print("MSE:", mse)

        print("RMSE:", rmse)

        print("R2:", r2)

        print("Score:", score)

        print("-" * 40)

In [9]:
#1º Experiência - BaseLine

X = df_linear[features]

y = df_linear[target]

run_linear_experiment(
    run_name="Experiment_1_Baseline",
    X=X,
    y=y,
    random_state=42,
    test_size=0.2
)

2026/05/19 09:34:19 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/05/19 09:34:19 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c

Experiment_1_Baseline
MAE: 23113.140556320854
MSE: 19013232822.658768
RMSE: 137888.4796589576
R2: 0.007214354848489757
Score: 0.007214354848489757
----------------------------------------


In [10]:
# Experiment 2 - Change Randomness
#Modificar o random_state (pequeno valor)

run_linear_experiment(
    run_name="Experiment_2_Change_Randomness",
    X=X,
    y=y,
    random_state=10,
    test_size=0.2
)

2026/05/19 09:34:33 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/19 09:34:33 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Lo

Experiment_2_Change_Randomness
MAE: 23243.74478496799
MSE: 10520718639.793186
RMSE: 102570.55444811238
R2: 0.013250662996733276
Score: 0.013250662996733276
----------------------------------------


In [11]:
## Experiment 3 - Larger Test Size
# Modificar o test_size (aumentar o valor)

run_linear_experiment(
    run_name="Experiment_3_Larger_Test_Set",
    X=X,
    y=y,
    random_state=42,
    test_size=0.4
)

2026/05/19 09:34:41 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/19 09:34:41 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Lo

Experiment_3_Larger_Test_Set
MAE: 22583.92898759037
MSE: 13827477212.39911
RMSE: 117590.29386985608
R2: 0.009680926891857844
Score: 0.009680926891857844
----------------------------------------


In [12]:
## Experiment 4 - Smaller Test Size
# Modificar o test_size (diminuir o valor)

run_linear_experiment(
    run_name="Experiment_4_Smaller_Test_Set",
    X=X,
    y=y,
    random_state=42,
    test_size=0.1
)

2026/05/19 09:34:49 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/19 09:34:49 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Lo

Experiment_4_Smaller_Test_Set
MAE: 21935.10336437841
MSE: 3000949902.2769
RMSE: 54780.92644595288
R2: 0.04291755283953613
Score: 0.04291755283953613
----------------------------------------


In [13]:
## Experiment 5 - Remove one feature
# Remover a feature "Age_Code"

X_exp5 = df_linear.drop(columns=["Age_Code", target])

run_linear_experiment(
    run_name="Experiment_5_Remove_One_Feature",
    X=X_exp5,
    y=y,
    random_state=42,
    test_size=0.2
)

2026/05/19 09:34:57 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/19 09:34:57 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Lo

Experiment_5_Remove_One_Feature
MAE: 23106.63427292942
MSE: 19013291350.84533
RMSE: 137888.69188894835
R2: 0.007211298769389529
Score: 0.007211298769389529
----------------------------------------


In [14]:
## Experiment 6 - Remove two features
# Remover as features "Age_Code" e "WorkExp"

X_exp6 = df_linear.drop(
    columns=["Age_Code", "WorkExp", target]
)

run_linear_experiment(
    run_name="Experiment_6_Remove_Two_Features",
    X=X_exp6,
    y=y,
    random_state=42,
    test_size=0.2
)

2026/05/19 09:35:04 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/19 09:35:04 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Lo

Experiment_6_Remove_Two_Features
MAE: 23647.94580865361
MSE: 19028825616.87281
RMSE: 137945.00939458743
R2: 0.00640016914909336
Score: 0.00640016914909336
----------------------------------------


In [15]:
## Experiment 7 - Use half of the data
# Usar apenas metade dos dados para treinar o modelo

run_linear_experiment(
    run_name="Experiment_7_Reduce_Dataset_Size",
    X=X,
    y=y,
    random_state=42,
    test_size=0.2,
    use_half_data=True
)

2026/05/19 09:35:10 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/19 09:35:10 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Lo

Experiment_7_Reduce_Dataset_Size
MAE: 21939.724631654917
MSE: 7078984533.146652
RMSE: 84136.70146343183
R2: 0.014512182977732047
Score: 0.014512182977732047
----------------------------------------


In [16]:
## Experiment 8 - Combine Changes
# Combinar as mudanças do Experiment 2 e do Experiment 3

run_linear_experiment(
    run_name="Experiment_8_Combine_Changes",
    X=X,
    y=y,
    random_state=10,
    test_size=0.4
)

2026/05/19 09:35:18 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/19 09:35:18 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Lo

Experiment_8_Combine_Changes
MAE: 22894.427748265036
MSE: 9950039371.53153
RMSE: 99749.88406775985
R2: 0.013262797541475413
Score: 0.013262797541475413
----------------------------------------


### Introdução

Depois da analisar os resultados no MlFlow, conseguimos verificar que o modelo 'Linear Regression' teve um desempenho fraco, porque os valores de R² ficaram muito próximos de 0. Por exemplo:
    -0.01
    -0.04
    -0.006
Isto significa que as features utilizadas explicam muito pouco da variação salarial. Isso indica que as variávies YearsCodePro_Num, WorkExp e Age_Code sozinhas não conseguem prever bem o salário anual. E a razão para isso é que o salário depende de muitos fatores como tecnologias, país, empresa, senioridade, remote work, etc...

### ANÁLISE DE EXPERIÊNCIAS

# Experiência 1 - Baseline

A experiência baseline apresentou RMSE elevado e R^2 muito baixo, isso demonstra que o modelo linear simples possui baixa capacidade preditiva para o problema salarial do dataset Stack Overflow.

# Experiência 2 - Change Randomness

Depois de modificar o valor do random_state para 10, verificamos que quase não existiu impacto nenhum, isso indica que o modelo é relativamente estável e os resultados não dependem muito da divisão aleatória.

# Experiência 3 - Larger Test Set

A experiência 3 foi modificar o valor do test_size. Neste caso a alteração foi de 0.2 para 0.4 (aumentando) e com isso verificamos que o desempenho piorou ligeiramente. Isso acontece porque o modelo ficou com menos dados para treino o que levou a uma redução da capacidade de aprendixagem

# Experiência 4 - Smaller Test Set

Nesta experiência, ao contrário da anterior (experiência 3) diminuimos o valor do test_size para 0.1. Esta foi a melhor experiência da Regressão Linear porque apresentou um menor RMSE e um maior R^2 (0.04 - apesar de muito baixo foi o melhor). Isso mostra que mais dados de treino melhoram o desempenho do modelo.

# Experiência 5 - Remove One Feature

Nesta experiência o objetivo era remover uma das features, no caso removemos a feature Age_Code e resultou numa piora do desempenho. O que isto nos sugere é que a idade possui alguma relevância na previsão salarial

# Experiência 6 - Remove Two Features

Esta como é óbvio, foi uma das experiências com piores resultados, porque ao utilizarmos menos variáveis o modelo perdeu informação importante, e consequentemente aumentou o erro.

# Experiência 7 - Reduce Dataset Size

Reduzir o dataset para metade, também foi uma das piores experiências feitas, porque prejudicou muito os resultados. Isso acontece porque o modelo teve acesso a menos exemplos de aprendizagem.

# Experiência 8 - Combine Changes

Aqui o que foi feito, foi mistuirar experiências. O que fizemos foi diminuir o random_state para 10 e aumentar o test_size para 0.4, resumidamente não houve melhorias. Isso demonstra que as alterações estruturais do dataset têm mais impacto do que pequenas alterações de divisão.

### Conclusão

A Regressão Linear não foi suficiente para modelar corretamente o salário anual. Os baixos valores de R^2 demonstram que a relação entre as variáveis e o salário não é puramente linear, ou seja, provavelmente modelos mais complexos vão apresentar melhores resultados, como o Ridge, Lasso ou até o Random_Forest.
